# Code LLM Forge — Complete Pipeline (Gemma 4 E4B)

> Fine-tune Gemma 4 E4B for code generation + tool calling using premium, execution-verified data.

| Section | What It Does | Est. Time |
|---------|-------------|-----------|
| 0. Install | Dependencies (Unsloth official method) | 2 min |
| 1. Data | Download & filter 80K samples from 4 datasets | 20 min |
| 2. Train | Single-stage mixed SFT (code + tools together) | ~2.5 hrs |
| 3. Benchmark | Custom zero-contamination eval | 30 min |
| 4. Export | GGUF for llama.cpp | 15 min |

**Total: ~3.5 hours on A100 80GB**

### Data Sources
- **NVIDIA OpenCodeInstruct** (30K) — execution-verified code, filtered from 5M to top quality
- **Magicoder-OSS-Instruct** (25K) — real GitHub code as seeds, proven to beat ChatGPT
- **Salesforce xLAM** (20K) — structured JSON tool calling (not text-based like Glaive)
- **Magicoder-Evol-Instruct** (5K) — evolved hard coding problems

### Key Changes from v1
- Single stage instead of two (avoids catastrophic forgetting)
- Structured JSON tool calls (xLAM) instead of text-based (Glaive)
- Execution-verified code data instead of unverified
- LoRA r=32, 2 epochs, eval split for overfitting detection

## 0. Install Dependencies

In [ ]:
import subprocess, sys

# Fix typing_extensions FIRST, then restart kernel if needed
result = subprocess.run([sys.executable, "-m", "pip", "install", "--upgrade", "typing_extensions"], 
                       capture_output=True, text=True)
print(result.stdout.split("\n")[-2] if result.stdout else "Done")

# Test if it works now
try:
    from typing_extensions import TypeIs
    print("typing_extensions OK — no restart needed")
except ImportError:
    print("ERROR: Restart kernel now! (Kernel → Restart Kernel, then Run All)")
    raise SystemExit("Restart kernel and re-run")

In [ ]:
%%capture
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth
else:
    import torch; v = re.match(r'[\d]{1,}\.[\d]{1,}', str(torch.__version__)).group(0)
    xformers = 'xformers==' + {'2.10':'0.0.34','2.9':'0.0.33.post1','2.8':'0.0.32.post2'}.get(v, "0.0.34")
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth_zoo bitsandbytes accelerate {xformers} peft trl triton unsloth
!pip install --no-deps transformers==5.5.0
!pip install torchcodec
!pip install --no-deps --upgrade timm  # Needed for Gemma 4 model loading
import torch; torch._dynamo.config.recompile_limit = 64;

In [ ]:
import torch, os, json, gc, time, random
from datetime import datetime

# Reduce memory fragmentation — critical for large-vocab models like Gemma 4
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

gpu = torch.cuda.get_device_name(0)
vram = torch.cuda.get_device_properties(0).total_mem / 1e9
print(f"GPU: {gpu} ({vram:.0f} GB)")
print(f"PyTorch: {torch.__version__}, CUDA: {torch.version.cuda}")

os.makedirs("/content/forge", exist_ok=True)
os.makedirs("/content/forge/training", exist_ok=True)
os.makedirs("/content/forge/benchmarks", exist_ok=True)
os.makedirs("/content/forge/gguf", exist_ok=True)
print("Directories ready")

## 1. Data Preparation

Download & filter from 4 premium sources into a single 80K mixed dataset:
- **OpenCodeInstruct** (30K) — filtered from 5M: only perfect execution + high LLM judgement
- **Magicoder-OSS-Instruct** (25K) — real GitHub code seeds, diverse problems
- **xLAM Function Calling** (20K) — structured JSON tool calls (not text-based!)
- **Magicoder-Evol-Instruct** (5K) — evolved hard problems for stretch

In [ ]:
from datasets import load_dataset

print("Downloading datasets...\n")

# 1. NVIDIA OpenCodeInstruct — 5M execution-verified code samples
# Filter to only perfect test scores (all unit tests pass)
print("[1/4] OpenCodeInstruct (filtering from 5M samples)...")
ds_oci = load_dataset("nvidia/OpenCodeInstruct", split="train", streaming=True)

oci_samples = []
seen = 0
for ex in ds_oci:
    seen += 1
    if seen % 500_000 == 0:
        print(f"  Scanned {seen:,} / ~5M ... kept {len(oci_samples):,}")
    try:
        score = float(ex.get("average_test_score", 0))
        if score < 1.0:
            continue
        # Basic quality check: response must be substantial
        output = ex.get("output", "")
        if not output or len(output) < 50:
            continue
    except (ValueError, TypeError):
        continue
    oci_samples.append(ex)
    if len(oci_samples) >= 60_000:  # collect 60K, we'll sample 30K later
        break

print(f"  Filtered to {len(oci_samples):,} high-quality samples from {seen:,} scanned")

# If we didn't get enough perfect scores, relax to >= 0.8
if len(oci_samples) < 30_000:
    print(f"  Not enough perfect scores, relaxing filter to >= 0.8...")
    ds_oci2 = load_dataset("nvidia/OpenCodeInstruct", split="train", streaming=True)
    for ex in ds_oci2:
        try:
            score = float(ex.get("average_test_score", 0))
            if score < 0.8 or score == 1.0:
                continue
            output = ex.get("output", "")
            if not output or len(output) < 50:
                continue
        except (ValueError, TypeError):
            continue
        oci_samples.append(ex)
        if len(oci_samples) >= 60_000:
            break
    print(f"  After relaxed filter: {len(oci_samples):,} samples")

# 2. Magicoder OSS-Instruct — real GitHub code as seeds
print("\n[2/4] Magicoder-OSS-Instruct-75K...")
ds_magic_oss = load_dataset("ise-uiuc/Magicoder-OSS-Instruct-75K", split="train")
print(f"  -> {len(ds_magic_oss):,} samples")

# 3. Function Calling — two sources combined for coverage
# Hermes: high quality structured JSON (11.5K)
# hypervariance/function-calling-sharegpt: cleaned Glaive with proper JSON (86K)
print("\n[3/4] Function Calling datasets...")
ds_hermes_fc = load_dataset("NousResearch/hermes-function-calling-v1", "func_calling_singleturn", split="train")
ds_hermes_multi = load_dataset("NousResearch/hermes-function-calling-v1", "func_calling", split="train")
ds_fc_sharegpt = load_dataset("hypervariance/function-calling-sharegpt", split="train")
print(f"  Hermes single-turn: {len(ds_hermes_fc):,}")
print(f"  Hermes multi-turn:  {len(ds_hermes_multi):,}")
print(f"  FC-ShareGPT:        {len(ds_fc_sharegpt):,}")

# 4. Magicoder Evol-Instruct — evolved hard problems
print("\n[4/4] Magicoder-Evol-Instruct-110K...")
ds_magic_evol = load_dataset("ise-uiuc/Magicoder-Evol-Instruct-110K", split="train")
print(f"  -> {len(ds_magic_evol):,} samples")

print("\nAll datasets loaded!")

In [ ]:
# ── Formatters ───────────────────────────────────────────────────────────────
# Each returns a list of chat messages or None if invalid

def fmt_opencodeinstruct(ex):
    """NVIDIA OpenCodeInstruct: input/output with code solutions."""
    q = ex.get("input", "")
    a = ex.get("output", "")
    if not q or not a or len(a) < 20:
        return None
    return [{"role": "user", "content": q}, {"role": "assistant", "content": a}]

def fmt_magicoder(ex):
    """Magicoder OSS/Evol: problem/solution pairs."""
    q = ex.get("problem", ex.get("instruction", ""))
    a = ex.get("solution", ex.get("response", ""))
    if not q or not a or len(a) < 20:
        return None
    return [{"role": "user", "content": q}, {"role": "assistant", "content": a}]

def fmt_hermes(ex):
    """NousResearch Hermes: ShareGPT with from/value fields."""
    convos = ex.get("conversations", [])
    if not convos or len(convos) < 2:
        return None
    role_map = {"system": "system", "human": "user", "gpt": "assistant", "tool": "tool"}
    msgs = []
    for turn in convos:
        role = role_map.get(turn.get("from", ""), None)
        if role is None:
            continue
        value = turn.get("value", "")
        if not value:
            continue
        msgs.append({"role": role, "content": value})
    return msgs if len(msgs) >= 2 else None

def fmt_fc_sharegpt(ex):
    """hypervariance/function-calling-sharegpt: cleaned Glaive in ShareGPT format."""
    convos = ex.get("conversations", [])
    if not convos or len(convos) < 2:
        return None
    role_map = {
        "system": "system",
        "human": "user",
        "assistant": "assistant",
        "function_response": "tool",
    }
    msgs = []
    for turn in convos:
        role = role_map.get(turn.get("from", ""), None)
        if role is None:
            continue
        value = turn.get("value", "")
        if not value:
            continue
        msgs.append({"role": role, "content": value})
    return msgs if len(msgs) >= 2 else None

# ── Build mixed dataset ──────────────────────────────────────────────────────
random.seed(42)

print("Formatting and mixing datasets...\n")

all_samples = []

# 1. OpenCodeInstruct — 30K from pre-filtered pool
random.shuffle(oci_samples)
count = 0
for ex in oci_samples[:30_000]:
    r = fmt_opencodeinstruct(ex)
    if r:
        all_samples.append({"conversations": r})
        count += 1
print(f"  OpenCodeInstruct: {count:,} samples")

# 2. Magicoder-OSS — 25K random
indices = list(range(len(ds_magic_oss)))
random.shuffle(indices)
count = 0
for i in indices[:25_000]:
    r = fmt_magicoder(ds_magic_oss[i])
    if r:
        all_samples.append({"conversations": r})
        count += 1
print(f"  Magicoder-OSS:    {count:,} samples")

# 3. Function Calling — all Hermes + 16K from FC-ShareGPT = ~20K total
count = 0
for ex in ds_hermes_fc:
    r = fmt_hermes(ex)
    if r:
        all_samples.append({"conversations": r})
        count += 1
for ex in ds_hermes_multi:
    r = fmt_hermes(ex)
    if r:
        all_samples.append({"conversations": r})
        count += 1
hermes_count = count

# Fill remaining tool calling budget from FC-ShareGPT
fc_target = 20_000 - hermes_count
indices = list(range(len(ds_fc_sharegpt)))
random.shuffle(indices)
fc_count = 0
for i in indices[:fc_target]:
    r = fmt_fc_sharegpt(ds_fc_sharegpt[i])
    if r:
        all_samples.append({"conversations": r})
        fc_count += 1
print(f"  Hermes FC:        {hermes_count:,} samples")
print(f"  FC-ShareGPT:      {fc_count:,} samples")
print(f"  Tool Calling Tot: {hermes_count + fc_count:,} samples")

# 4. Magicoder-Evol — 5K random
indices = list(range(len(ds_magic_evol)))
random.shuffle(indices)
count = 0
for i in indices[:5_000]:
    r = fmt_magicoder(ds_magic_evol[i])
    if r:
        all_samples.append({"conversations": r})
        count += 1
print(f"  Magicoder-Evol:   {count:,} samples")

# Shuffle the full mix
random.shuffle(all_samples)
print(f"\n  TOTAL: {len(all_samples):,} samples")

# Save as JSONL
path = "/content/forge/train_data.jsonl"
with open(path, "w") as f:
    for item in all_samples:
        f.write(json.dumps(item) + "\n")
print(f"  Saved to {path}")

# Free memory
del oci_samples, ds_magic_oss, ds_hermes_fc, ds_hermes_multi, ds_fc_sharegpt, ds_magic_evol
gc.collect()
print("  Datasets freed from memory")

## 2. Training — Single-Stage Mixed SFT

Single stage with all data mixed together (code + tool calling).
This avoids catastrophic forgetting from multi-stage training.

**Config:**
- LoRA rank 32 (2x previous — more capacity for code tasks)
- Learning rate 1e-4 (half of v1 — less aggressive, preserves base capabilities)
- 2 epochs (v1 was 1 — underfitting at 80K samples)
- 5% eval split for overfitting detection
- `train_on_responses_only` — don't waste compute on user prompts

In [ ]:
from unsloth import FastModel

model, tokenizer = FastModel.from_pretrained(
    model_name = "unsloth/gemma-4-E4B-it",
    max_seq_length = 4096,
    dtype = None,
    load_in_4bit = True,
    full_finetuning = False,
)
print("Model loaded!")

model = FastModel.get_peft_model(
    model,
    finetune_vision_layers     = False,
    finetune_language_layers   = True,
    finetune_attention_modules = True,
    finetune_mlp_modules       = True,
    r = 32,            # 2x v1 — more capacity for code generation
    lora_alpha = 64,   # 2x rank (community consensus for Gemma 4)
    lora_dropout = 0.05,  # light regularization
    bias = "none",
    random_state = 3407,
)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"Trainable: {trainable:,} / {total:,} ({100*trainable/total:.2f}%)")

In [ ]:
from unsloth.chat_templates import get_chat_template, standardize_data_formats
from datasets import Dataset

# Apply Gemma 4 chat template
tokenizer = get_chat_template(tokenizer, chat_template="gemma-4")

# Load mixed training data
with open("/content/forge/train_data.jsonl") as f:
    records = [json.loads(l) for l in f]

# Remap "tool" role to "assistant" — Unsloth's standardize_data_formats
# doesn't recognize "tool" as a valid role
for rec in records:
    for msg in rec.get("conversations", []):
        if msg.get("role") == "tool":
            msg["role"] = "assistant"
            # Prefix so the model knows this is a tool response
            msg["content"] = f"[Tool Response]\n{msg['content']}"

ds = Dataset.from_list(records)
ds = standardize_data_formats(ds)
print(f"Full dataset: {len(ds):,} samples")

# Apply chat template
def format_prompts(examples):
    convos = examples["conversations"]
    texts = []
    for convo in convos:
        try:
            t = tokenizer.apply_chat_template(convo, tokenize=False, add_generation_prompt=False)
            texts.append(t.removeprefix('<bos>'))
        except:
            texts.append("")
    return {"text": texts}

ds = ds.map(format_prompts, batched=True, num_proc=4)
ds = ds.filter(lambda x: len(x["text"]) > 50)
print(f"After formatting: {len(ds):,} samples")

# Train/eval split (95/5)
split = ds.train_test_split(test_size=0.05, seed=42)
ds_train = split["train"]
ds_eval = split["test"]
print(f"Train: {len(ds_train):,} | Eval: {len(ds_eval):,}")
print(f"\nSample:\n{ds_train[0]['text'][:400]}")

In [ ]:
from trl import SFTTrainer, SFTConfig
from unsloth.chat_templates import train_on_responses_only

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = ds_train,
    eval_dataset = ds_eval,
    args = SFTConfig(
        dataset_text_field = "text",
        per_device_train_batch_size = 2,   # small — Gemma 4 262K vocab eats VRAM on logits
        gradient_accumulation_steps = 16,  # effective batch = 32 (2 × 16)
        num_train_epochs = 2,
        learning_rate = 1e-4,
        warmup_steps = 100,
        logging_steps = 25,
        save_strategy = "steps",
        save_steps = 500,
        save_total_limit = 2,
        eval_strategy = "steps",
        eval_steps = 500,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "cosine",
        seed = 3407,
        output_dir = "/content/forge/training",
        report_to = "none",
        max_seq_length = 4096,
        bf16 = True,
        dataloader_num_workers = 4,
        load_best_model_at_end = True,
        metric_for_best_model = "eval_loss",
    ),
)

trainer = train_on_responses_only(
    trainer,
    instruction_part = "<|turn>user\n",
    response_part = "<|turn>model\n",
)

print("Starting training...")
print(f"  Samples: {len(ds_train):,} train + {len(ds_eval):,} eval")
print(f"  Effective batch: 32 (2 × 16) | Epochs: 2 | LR: 1e-4")
print(f"  LoRA: r=32, alpha=64, dropout=0.05")
t0 = time.time()
stats = trainer.train()
train_time = time.time() - t0
print(f"\nTraining done! {train_time/3600:.2f} hrs, final loss={stats.training_loss:.4f}")

In [ ]:
# Save merged model
print("Saving merged model...")
model.save_pretrained_merged("/content/forge/training/merged", tokenizer)
print("Model saved!")

print(f"\n{'='*50}")
print(f"TRAINING COMPLETE")
print(f"Time: {train_time/3600:.2f} hrs")
print(f"Final loss: {stats.training_loss:.4f}")
print(f"{'='*50}")

del model, trainer, ds_train, ds_eval
gc.collect()
torch.cuda.empty_cache()
print(f"GPU memory freed: {torch.cuda.memory_allocated()/1e9:.1f} GB used")

In [ ]:
from unsloth import FastModel
from unsloth.chat_templates import get_chat_template

def load_model(model_name):
    m, tok = FastModel.from_pretrained(
        model_name = model_name,
        max_seq_length = 2048,
        dtype = None,
        load_in_4bit = True,
    )
    tok = get_chat_template(tok, chat_template="gemma-4")
    return m, tok

def ask(model, tokenizer, prompt, max_tokens=512):
    msgs = [{"role":"user","content":[{"type":"text","text":prompt}]}]
    inputs = tokenizer.apply_chat_template(msgs, add_generation_prompt=True, return_tensors="pt", tokenize=True, return_dict=True).to("cuda")
    out = model.generate(**inputs, max_new_tokens=max_tokens, temperature=1.0, top_p=0.95, top_k=64, use_cache=True)
    return tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)

MODELS = {
    "Base (untuned)":  "unsloth/gemma-4-E4B-it",
    "Fine-tuned":      "/content/forge/training/merged",
}
print("Models to evaluate:", list(MODELS.keys()))

In [ ]:
# Export to GGUF (Q8_0 — best quality Gemma 4 supports)
try:
    del model
except NameError:
    pass
gc.collect(); torch.cuda.empty_cache()

from unsloth import FastModel
model, tokenizer = FastModel.from_pretrained(
    model_name = "/content/forge/training/merged",
    max_seq_length = 2048,
    dtype = None,
    load_in_4bit = False,  # Need full precision for GGUF
)

print("Exporting Q8_0 GGUF...")
model.save_pretrained_gguf(
    "/content/forge/gguf",
    tokenizer,
    quantization_method = "Q8_0",
)
print("\nGGUF export done!")

# List files
for f in sorted(os.listdir("/content/forge/gguf")):
    if f.endswith(".gguf"):
        sz = os.path.getsize(f"/content/forge/gguf/{f}") / 1e9
        print(f"  {f}: {sz:.2f} GB")

In [ ]:
# ============================================================
# DONE!
# ============================================================
print("=" * 50)
print("CODE LLM FORGE v2 — COMPLETE!")
print("=" * 50)
try:
    print(f"Training: {train_time/3600:.2f} hrs, loss={stats.training_loss:.4f}")
except NameError:
    print("(Timing variables not available)")
print(f"GGUF: /content/forge/gguf/")
print()
print("To download:")
print("  from google.colab import files")
print("  files.download('/content/forge/gguf/YOUR_FILE.gguf')")
print()
print("To run locally:")
print("  llama-server -m model-Q8_0.gguf --n-gpu-layers 99 -n -1")
print()
print("To benchmark locally:")
print("  python run_all.py --model finetuned --quick-only")
print("  python run_all.py --compare")